# 12a — Validation Evaluation

## Requirements covered

This notebook:

- loads the tuned Week 5 candidate models;
- evaluates them on the held-out validation datasets;
- compares validation performance across datasets/models;
- generates validation confusion matrices;
- selects the best candidate model(s).

### Outputs

- `outputs/metrics/validation_results.csv`
- validation confusion-matrix `.csv` and `.png` files
- `outputs/tables/validation_comparison.csv`
- `outputs/tables/validation_best_models.csv`
- `outputs/figures/validation_model_comparison.png`

**Selection rule:** Macro F1 is the primary metric, with balanced accuracy used as the tie-breaker.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import warnings
import re

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 100)

# Find project root whether the notebook is run from the repo root or /notebooks.
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the project repository.")

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for folder in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"
PRIMARY_METRIC = "macro_f1"
TIE_BREAKER = "balanced_accuracy"

print("Project root:", PROJECT_ROOT)

## 2. Load Week 5 candidate models

The Week 5 candidate table is used when available. If it does not contain saved model paths, the notebook searches the project `models/` and `outputs/` folders for saved `.joblib`, `.pkl`, or `.pickle` models.

In [ ]:
CANDIDATE_FILE = METRICS_DIR / "candidate_models.csv"

def first_matching_column(columns, options):
    lookup = {str(c).lower(): c for c in columns}
    for option in options:
        if option.lower() in lookup:
            return lookup[option.lower()]
    return None

def clean_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")

# Start with Week 5 candidate table when present.
candidate_table = pd.read_csv(CANDIDATE_FILE) if CANDIDATE_FILE.exists() else pd.DataFrame()

dataset_col = first_matching_column(
    candidate_table.columns,
    ["dataset", "dataset_name", "modality", "feature_set"]
) if not candidate_table.empty else None

model_col = first_matching_column(
    candidate_table.columns,
    ["model", "model_name", "classifier", "estimator"]
) if not candidate_table.empty else None

path_col = first_matching_column(
    candidate_table.columns,
    ["model_file", "model_path", "artifact_path", "saved_model", "path"]
) if not candidate_table.empty else None

model_records = []

# Use paths recorded in candidate_models.csv when available.
if not candidate_table.empty and path_col:
    for _, row in candidate_table.iterrows():
        raw_path = row[path_col]
        if pd.isna(raw_path):
            continue

        p = Path(str(raw_path))
        if not p.is_absolute():
            p = PROJECT_ROOT / p

        if p.exists():
            model_records.append({
                "dataset": str(row[dataset_col]) if dataset_col else p.stem,
                "model": str(row[model_col]) if model_col else p.stem,
                "model_file": str(p),
            })

# Fallback: discover saved Week 5 model files.
if not model_records:
    search_roots = [MODELS_DIR, OUTPUTS_DIR]
    for root in search_roots:
        if not root.exists():
            continue
        for ext in ("*.joblib", "*.pkl", "*.pickle"):
            for p in root.rglob(ext):
                model_records.append({
                    "dataset": p.stem,
                    "model": p.stem,
                    "model_file": str(p),
                })

model_inventory = (
    pd.DataFrame(model_records)
    .drop_duplicates(subset=["model_file"])
    .reset_index(drop=True)
)

if model_inventory.empty:
    raise FileNotFoundError(
        "No saved Week 5 model artifacts were found. "
        "Check outputs/metrics/candidate_models.csv or the models/ folder."
    )

display(model_inventory)
print(f"Models found: {len(model_inventory)}")

## 3. Match each model to its validation dataset

In [ ]:
# Search only for CSV files that clearly look like validation datasets.
validation_files = []
for root in [DATA_DIR, OUTPUTS_DIR]:
    if root.exists():
        validation_files.extend(root.rglob("*validation*.csv"))

# Do not accidentally use this notebook's own output files as inputs.
validation_files = [
    p for p in validation_files
    if "validation_results" not in p.name.lower()
    and "comparison" not in p.name.lower()
    and "confusion" not in p.name.lower()
]

def best_validation_match(dataset_name):
    if not validation_files:
        return None

    ds = clean_name(dataset_name)
    ds_tokens = set(ds.split("_"))

    scored = []
    for p in validation_files:
        name = clean_name(p.stem)
        name_tokens = set(name.split("_"))

        # Prefer files containing the dataset name; otherwise use token overlap.
        exact_bonus = 100 if ds and ds in name else 0
        overlap = len(ds_tokens & name_tokens)
        scored.append((exact_bonus + overlap, p))

    scored.sort(key=lambda x: x[0], reverse=True)

    # If only one validation dataset exists, use it.
    if len(validation_files) == 1:
        return validation_files[0]

    return scored[0][1] if scored and scored[0][0] > 0 else None

model_inventory["validation_file"] = model_inventory["dataset"].apply(best_validation_match)

display(model_inventory[["dataset", "model", "model_file", "validation_file"]])

missing = model_inventory["validation_file"].isna().sum()
if missing:
    print(
        f"WARNING: {missing} model(s) could not be matched to a validation CSV. "
        "If needed, edit the validation_file values in model_inventory before continuing."
    )

## 4. Evaluate all models

No model is re-trained here. Each saved tuned model is loaded and used only to predict the held-out validation data.

In [ ]:
EXCLUDED_COLUMNS = {
    ID_COLUMN,
    TARGET,
    "condition_group",
    "condition_original",
}

def load_estimator(path):
    obj = joblib.load(path)

    # Allow a saved estimator directly or a dictionary containing one.
    if hasattr(obj, "predict"):
        return obj

    if isinstance(obj, dict):
        for key in ["model", "estimator", "pipeline", "best_estimator"]:
            if key in obj and hasattr(obj[key], "predict"):
                return obj[key]

    raise TypeError(f"Saved artifact does not contain a prediction-capable estimator: {path}")

def prepare_X(model, df):
    feature_cols = [c for c in df.columns if c not in EXCLUDED_COLUMNS]
    X = df[feature_cols].copy()

    # If sklearn stored the original feature names, align to them.
    if hasattr(model, "feature_names_in_"):
        required = list(model.feature_names_in_)
        missing = [c for c in required if c not in X.columns]
        if missing:
            raise ValueError(f"Validation dataset is missing model features: {missing[:10]}")
        X = X[required]

    return X

results = []
predictions = {}

for _, row in model_inventory.dropna(subset=["validation_file"]).iterrows():
    dataset = row["dataset"]
    model_name = row["model"]
    model_path = Path(row["model_file"])
    validation_path = Path(row["validation_file"])

    try:
        df = pd.read_csv(validation_path, dtype={ID_COLUMN: str})

        if TARGET not in df.columns:
            raise ValueError(f"'{TARGET}' column not found in {validation_path.name}")

        model = load_estimator(model_path)
        X_val = prepare_X(model, df)
        y_true = df[TARGET]
        y_pred = model.predict(X_val)

        key = f"{clean_name(dataset)}__{clean_name(model_name)}"
        predictions[key] = (y_true, y_pred)

        results.append({
            "dataset": dataset,
            "model": model_name,
            "n_validation": len(df),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "model_file": str(model_path),
            "validation_file": str(validation_path),
            "result_key": key,
        })

        print(f"✓ {dataset} | {model_name}")

    except Exception as e:
        print(f"✗ {dataset} | {model_name}: {e}")

validation_results = pd.DataFrame(results)

if validation_results.empty:
    raise RuntimeError(
        "No models were evaluated successfully. "
        "Review the printed errors above."
    )

validation_results = validation_results.sort_values(
    [PRIMARY_METRIC, TIE_BREAKER],
    ascending=False,
).reset_index(drop=True)

validation_results.insert(0, "rank", range(1, len(validation_results) + 1))
display(validation_results)

## 5. Save validation results and comparison table

In [ ]:
validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

comparison_columns = [
    "rank",
    "dataset",
    "model",
    "n_validation",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

validation_comparison = validation_results[comparison_columns].copy()
validation_comparison.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

display(validation_comparison.round(4))

## 6. Validation confusion matrices

In [ ]:
for _, row in validation_results.iterrows():
    key = row["result_key"]
    y_true, y_pred = predictions[key]

    labels = sorted(pd.Series(y_true).dropna().unique().tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Save matrix as CSV.
    cm_df = pd.DataFrame(
        cm,
        index=[f"actual_{x}" for x in labels],
        columns=[f"predicted_{x}" for x in labels],
    )
    cm_df.to_csv(METRICS_DIR / f"{key}_validation_confusion_matrix.csv")

    # Save and display matrix as figure.
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels,
    ).plot(ax=ax, values_format="d")
    ax.set_title(f"{row['dataset']} — {row['model']}")
    fig.tight_layout()
    fig.savefig(
        FIGURES_DIR / f"{key}_validation_confusion_matrix.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

## 7. Validation comparison figure

In [ ]:
plot_df = validation_results.copy()
plot_df["candidate"] = (
    plot_df["dataset"].astype(str)
    + " | "
    + plot_df["model"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(plot_df))))
ax.barh(plot_df["candidate"], plot_df["macro_f1"])
ax.set_xlabel("Validation Macro F1")
ax.set_ylabel("Dataset | Model")
ax.set_title("Validation Model Comparison")
ax.invert_yaxis()
fig.tight_layout()

fig.savefig(
    FIGURES_DIR / "validation_model_comparison.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

## 8. Select the best model(s)

In [ ]:
# Best candidate within each dataset.
best_by_dataset = (
    validation_results
    .sort_values(
        ["dataset", PRIMARY_METRIC, TIE_BREAKER],
        ascending=[True, False, False],
    )
    .groupby("dataset", as_index=False)
    .first()
)

best_by_dataset.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

display(
    best_by_dataset[
        ["dataset", "model", "macro_f1", "balanced_accuracy", "accuracy"]
    ].round(4)
)

# Overall best candidate.
best = validation_results.iloc[0]

print(
    f"BEST OVERALL CANDIDATE: {best['model']} "
    f"({best['dataset']})"
)
print(f"Macro F1: {best['macro_f1']:.4f}")
print(f"Balanced accuracy: {best['balanced_accuracy']:.4f}")

## 9. Validation metrics summary and justification

In [ ]:
summary = validation_results[
    ["dataset", "model", "accuracy", "balanced_accuracy", "macro_f1"]
].copy()

display(summary.round(4))

print("\nJUSTIFICATION")
print(
    f"{best['model']} on {best['dataset']} is the strongest validation candidate "
    f"because it achieved the highest Macro F1 ({best['macro_f1']:.4f}). "
    f"Its balanced accuracy was {best['balanced_accuracy']:.4f}, which is used "
    "as the tie-breaker when Macro F1 scores are close. "
    "The best-performing model for each dataset is also retained in "
    "validation_best_models.csv for final model review."
)

## 10. Deliverables check

In [ ]:
deliverables = pd.DataFrame([
    {
        "deliverable": "validation_results.csv",
        "path": METRICS_DIR / "validation_results.csv",
    },
    {
        "deliverable": "Validation comparison table",
        "path": TABLES_DIR / "validation_comparison.csv",
    },
    {
        "deliverable": "Best models table",
        "path": TABLES_DIR / "validation_best_models.csv",
    },
    {
        "deliverable": "Validation comparison figure",
        "path": FIGURES_DIR / "validation_model_comparison.png",
    },
])

deliverables["status"] = deliverables["path"].apply(
    lambda p: "READY" if Path(p).exists() else "MISSING"
)

display(deliverables)

print(
    "Confusion-matrix CSVs:",
    len(list(METRICS_DIR.glob("*_validation_confusion_matrix.csv"))),
)
print(
    "Confusion-matrix figures:",
    len(list(FIGURES_DIR.glob("*_validation_confusion_matrix.png"))),
)